In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import numpy as np
import pathlib
import torch
import torch.nn as nn
from scipy.interpolate import interp1d

DEVICE = torch.device("cuda")
BASE = pathlib.Path.home() / "taekwondo-scorer"
NORMALIZED_DIR = BASE / "processed/normalized"
TEMPLATES_DIR  = BASE / "templates"
MODEL_DIR      = BASE / "models"

ACTION_NAMES = [
    "기본준비", "앞굽이하고 아래막고 지르기", "앞굽이하고 아래막기",
    "앞굽이하고 지르기", "앞서고 아래막기", "앞서고 안막기",
    "앞서고 얼굴막기", "앞서고 지르기", "앞차고 앞서고 지르기"
]
JOINT_NAMES = ["왼팔꿈치", "오른팔꿈치", "왼어깨", "오른어깨",
               "왼무릎", "오른무릎", "왼엉덩이", "오른엉덩이"]

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(8, 64, 2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(64, 32)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1])

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(32, 64)
        self.lstm = nn.LSTM(64, 64, 2, batch_first=True, dropout=0.2)
        self.out = nn.Linear(64, 8)
    def forward(self, z):
        x = self.fc(z).unsqueeze(1).repeat(1, 60, 1)
        out, _ = self.lstm(x)
        return self.out(out)

class LSTMAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()
    def forward(self, x):
        return self.decoder(self.encoder(x))

# 모델 + 템플릿 + 통계 로딩
ae_models, templates = {}, {}
with open(BASE / "distance_stats.json") as f:
    dtw_stats = json.load(f)
with open(BASE / "error_stats.json") as f:
    error_stats = json.load(f)

for action in ACTION_NAMES:
    m = LSTMAutoencoder().to(DEVICE)
    m.load_state_dict(torch.load(MODEL_DIR / f"{action}_autoencoder.pth", map_location=DEVICE))
    m.eval()
    ae_models[action] = m
    templates[action] = np.load(TEMPLATES_DIR / f"{action}.npy")

print(f"✅ 모델 {len(ae_models)}개 + 템플릿 로딩 완료!")

# 함수 정의
def resample(seq, n=60):
    T = len(seq)
    if T == 1:
        return np.tile(seq, (n, 1))
    f = interp1d(np.linspace(0,1,T), seq, axis=0)
    return f(np.linspace(0,1,n))

def calc_score(v, s):
    mn,p10,p25,p50,p75,p90,mx = s["min"],s["p10"],s["p25"],s["p50"],s["p75"],s["p90"],s["max"]
    if v<=mn: return 100.0
    elif v<=p10: return 85+15*(1-(v-mn)/(p10-mn+1e-8))
    elif v<=p25: return 70+15*(1-(v-p10)/(p25-p10+1e-8))
    elif v<=p50: return 55+15*(1-(v-p25)/(p50-p25+1e-8))
    elif v<=p75: return 40+15*(1-(v-p50)/(p75-p50+1e-8))
    elif v<=p90: return 20+20*(1-(v-p75)/(p90-p75+1e-8))
    else: return max(0,20*(1-(v-p90)/(mx-p90+1e-8)))

def score_with_model(seq, action_name):
    tensor = torch.tensor(resample(seq,60).astype(np.float32)/180.0).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        output = ae_models[action_name](tensor)
    recon_error = nn.functional.mse_loss(output, tensor).item()
    score = calc_score(recon_error, error_stats[action_name])
    joint_err = torch.abs(output[0,-1]-tensor[0,-1]).cpu().numpy()
    joint_errors = {n:round(float(e*180),2) for n,e in zip(JOINT_NAMES, joint_err)}
    worst = max(joint_errors, key=joint_errors.get)
    return score, joint_errors, recon_error, worst

print("✅ 함수 정의 완료!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
from scipy.interpolate import interp1d
import time

NORMALIZED_DIR = pathlib.Path.home() / "taekwondo-scorer/processed/normalized"

# resample 함수 정의 (여기서 빠졌던 것!)
def resample(seq, n=60):
    T = len(seq)
    if T == 1:
        return np.tile(seq, (n, 1))
    f = interp1d(np.linspace(0, 1, T), seq, axis=0)
    return f(np.linspace(0, 1, n))

action_name = "앞서고 지르기"
all_files = list((NORMALIZED_DIR / action_name).glob("*/*.npy"))[:50]

dtw_scores, lstm_scores, final_scores = [], [], []
latency_dtw, latency_lstm = [], []

print(f"=== {action_name} 50개 샘플 비교 ===\n")

for f in all_files:
    seq = np.load(f)

    # DTW 채점
    t0 = time.time()
    resampled = resample(seq, 60).astype(np.float32)
    dtw_dist, _ = fastdtw(resampled, templates[action_name], dist=euclidean)
    dtw_score = calc_score(dtw_dist, dtw_stats[action_name])
    latency_dtw.append(time.time() - t0)

    # LSTM 채점
    t0 = time.time()
    lstm_score, _, recon_err, _ = score_with_model(seq, action_name)
    latency_lstm.append(time.time() - t0)

    final = lstm_score * 0.6 + dtw_score * 0.4
    dtw_scores.append(dtw_score)
    lstm_scores.append(lstm_score)
    final_scores.append(final)

dtw_scores   = np.array(dtw_scores)
lstm_scores  = np.array(lstm_scores)
final_scores = np.array(final_scores)

print("📊 점수 분포 비교:")
print(f"{'방식':<12} {'평균':>8} {'최고':>8} {'최저':>8} {'표준편차':>10} {'속도(ms)':>10}")
print("-" * 60)
print(f"{'DTW':<12} {dtw_scores.mean():>8.1f} {dtw_scores.max():>8.1f} {dtw_scores.min():>8.1f} {dtw_scores.std():>10.1f} {np.mean(latency_dtw)*1000:>10.1f}")
print(f"{'LSTM':<12} {lstm_scores.mean():>8.1f} {lstm_scores.max():>8.1f} {lstm_scores.min():>8.1f} {lstm_scores.std():>10.1f} {np.mean(latency_lstm)*1000:>10.1f}")
print(f"{'LSTM+DTW':<12} {final_scores.mean():>8.1f} {final_scores.max():>8.1f} {final_scores.min():>8.1f} {final_scores.std():>10.1f} {'(합산)':>10}")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(dtw_scores, lstm_scores, alpha=0.6, color='steelblue')
axes[0].plot([0,100],[0,100], 'r--', alpha=0.5, label='y=x')
axes[0].set_xlabel('DTW 점수')
axes[0].set_ylabel('LSTM 점수')
axes[0].set_title('DTW vs LSTM 점수 비교')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(dtw_scores,  bins=15, alpha=0.5, label='DTW',      color='blue')
axes[1].hist(lstm_scores, bins=15, alpha=0.5, label='LSTM',     color='orange')
axes[1].hist(final_scores,bins=15, alpha=0.5, label='LSTM+DTW', color='green')
axes[1].set_xlabel('점수')
axes[1].set_ylabel('빈도')
axes[1].set_title('점수 분포')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ 최종 결정: LSTM 60% + DTW 40% 결합 방식 채택")
print("이유: 두 방식의 장점을 결합하여 더 안정적인 채점 가능")